# CSDesign — Conformationally-Specific Protein Design

**HOW TO RUN**
1. Install the dependencies
2. Download the PDBs. In line 5 of the code block there is a list of PDBs to download, modify it to download the desired PDBs
3. Verify GPU
4. Run CSDesign. Currently there are 4 examples from the paper to show syntax. Here is a quick explanation of the flags:

**protein_id** [PDB]: The structure that CSDesign will optimize the sequence to fold into.

**protein_id_anti** [PDB]: The structure that CSDesign will optimize the sequence to not fold into.

**decode_order** [n_to_c, proximity, reverse_proximity, random]:
- n_to_c: Predicts residues in order from N-terminus to C-terminus.
- proximity: Predicts residues closest to the fixed positions first, working outward.
- reverse_proximity: Predicts residues farthest from the fixed positions first, working inward.
- random: Predicts residues in a random order.

**decode_algorithm** [greedy, beam]: The algorithm used to generate the sequence. Beam search (default) tracks multiple candidate sequences in parallel and generally gives better results. Greedy picks the single best residue at each step.

**fixed_positions** [int int ...]: Selects positions that are unable to mutate, specified as pairs of PDB residue numbers defining inclusive ranges. An input of "1 168 187 358" means residues 1–168 and 187–358 are fixed. Note these are PDB residue numbers from the ATOM records, not positional indices in the file.

**ball_mask**: Expands the mutable region. Takes the positions already allowed to mutate (the unfixed positions from --fixed_positions) and additionally unfixes any residue within 8Å of those positions. Overrides the fixed_positions mask for any residue that falls within that radius.

**balance_factor** [float]: Dampens extreme probability ratios in the anti-protein scoring (default 0.002).




## 1 · Clone repo & install dependencies

Run this cell once per session.

In [ ]:
!git clone --depth 1 https://github.com/dellacortelab/cs_design.git
%cd cs_design

!pip install torch --quiet

!pip install \
    numpy==2.1.1 \
    biopython==1.81 \
    sentencepiece==0.2.0 \
    transformers==4.44.2 \
    tokenizers==0.19.1 \
    scipy==1.15.1 \
    accelerate==1.3.0 \
    pandas==2.2.3 \
    --quiet

!pip install -e . --quiet

print('\u2705 Setup complete.')

Cloning into 'cs_design'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 37 (delta 0), reused 27 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 24.20 MiB | 19.43 MiB/s, done.
/content/cs_design
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 M

## 2 · Download PDB structures

The script expects PDB files at `./data/structures/<ID>.pdb`.
This cell downloads **4GSB** and **2ERK** directly from the RCSB PDB.

In [ ]:
import os, urllib.request
os.makedirs('data/structures', exist_ok=True)


pdb_ids = ['4GSB', '2ERK']


for pdb_id in pdb_ids:
    dest = f'data/structures/{pdb_id}.pdb'
    if os.path.exists(dest):
        print(f'{pdb_id}.pdb already present, skipping.')
        continue
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    print(f'Downloading {pdb_id} from RCSB...', end=' ')
    urllib.request.urlretrieve(url, dest)
    print('done.')

print('\u2705 PDB files ready.')

## 3 · Verify GPU

In [ ]:
# import torch
# print('PyTorch version:', torch.__version__)
# print('CUDA available: ', torch.cuda.is_available())
# if torch.cuda.is_available():
#     print('GPU:            ', torch.cuda.get_device_name(0))

## 4 · Run experiments

In [ ]:
# CSD101 — 4GSB → anti 2ERK, no ball_mask
!python3 src/cs_design/design.py \
    --model_name cs_design \
    --protein_id 4GSB \
    --protein_id_anti 2ERK \
    --decode_order n_to_c \
    --decode_algorithm greedy \
    --fixed_positions 1 168 187 358

In [ ]:
# CSD102 — 4GSB → anti 2ERK, with --ball_mask
!python3 src/cs_design/design.py \
    --model_name cs_design \
    --protein_id 4GSB \
    --protein_id_anti 2ERK \
    --decode_order n_to_c \
    --decode_algorithm greedy \
    --fixed_positions 1 168 187 358 \
    --ball_mask

In [ ]:
# CSD103 — 2ERK → anti 4GSB, no ball_mask
!python3 src/cs_design/design.py \
    --model_name cs_design \
    --protein_id 2ERK \
    --protein_id_anti 4GSB \
    --decode_order n_to_c \
    --decode_algorithm greedy \
    --fixed_positions 1 168 187 358

In [ ]:
# CSD104 — 2ERK → anti 4GSB, with --ball_mask
!python3 src/cs_design/design.py \
    --model_name cs_design \
    --protein_id 2ERK \
    --protein_id_anti 4GSB \
    --decode_order n_to_c \
    --decode_algorithm greedy \
    --fixed_positions 1 168 187 358 \
    --ball_mask